# Max-basis arm B (s=1.41) on NSTX_MTM n300, MTM-only (Fusion_PhD-bhx7.12)

User, 2026-10-02: "This needs to be an MTM database, we should be using the rules I established to remove anything that isn't an MTM." (`docs/mode_filtering.md`, "Populations".)

**Population.** NSTX_MTM GS2 cases with $\gamma_{GS2} > 10^{-3}$ whose GS2 dominant mode `general_analysis.mode_classification.classify` labels MTM ($T > 0.15$ and $\omega < 0$; `Fusion_PhD-lqwx`, GA master `77e04d9`). Everything else (KBM, mixed, GS2-stable) is removed and the count is printed.

**Scoring the code (user, 2026-10-02).** GFTM is never filtered. Its dominant mode (argmax $\gamma$ over `mode`) is scored whatever its type. *Capture* = that mode is an MTM by $T>0.15$ and $\omega<0$. A dominant mode that is not an MTM is a capture failure whose growth-rate error still counts in medlog / medAE. No unstable mode, or no deck, is a MISS ($+\infty$), kept.

GFTM's $T$ is the pyro formula $|\int A_\parallel d\theta| / \int |A_\parallel| d\theta$ on the stored theta-resolved `apar` of the dominant mode (unit $dl$ weight, Simpson). For the two max-basis arms the stored `tearing_parameter` exists; the agreement is checked below.

Scored beside the existing n300 arms from `mtm_only_rescore`. Everything is read from each leaf's `pyroscan.nc` through `mc.load_gs2_cube` / `mc.load_gftm_cube`; no run directory is opened.

## Imports and settings

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from scipy.integrate import simpson
from general_analysis import mode_classification as mc
from general_analysis.mode_classification import T_TEAR

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
plt.rcParams.update({"axes.titlesize": 16, "axes.labelsize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14, "legend.fontsize": 12})
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "mtm_maxbasis_n300"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
base = data_root / "GFTM/Runs/LATIN_HYPERCUBE/NSTX_MTM"
gs2_base = data_root / "GS2/Runs/LATIN_HYPERCUBE/NSTX_MTM"

# label -> (leaf, folder with the cube, file kind)
arms_all = {
    "A maxbasis s=1":    ("extent_maxbasis_s100", "", "gyrorun"),
    "B maxbasis s=1.41": ("extent_maxbasis_s141", "", "gyrorun"),
    "m11_new":           ("m11_new", "pyro_cube", "cube"),
    "m9_new":            ("m9_new", "pyro_cube", "cube"),
    "rule s=1":          ("extent_rule_tf035", "pyro_cube", "cube"),
    "rule s=1.41":       ("extent_rule_tf035_s141", "pyro_cube", "cube"),
    "fw174":             ("gftm_fw174", "pyro_cube", "cube"),
}
# n300 only (Fusion_PhD-bhx7.12). Arm B is the new leaf; arm A was never built on n300.
arms_all.pop("A maxbasis s=1")
draws = {"n300": ("beta_q_shat_ky_n300", list(arms_all))}
gs2_unstable = 1e-3
n_boot, seed = 2000, 0

## Load and score

Per draw: the GS2 cube (`pyro_cube_avg`, tail-averaged $\gamma$ as in every earlier analysis), the GS2 classifier label per GS2-unstable case (`mc.indicators(scan, ds)` on the same tail-averaged cube, no run directory; `lqwx` used each run's final time), and every GFTM arm's dominant-mode $\gamma$, $\omega$, $T$ (pyro's `FieldLine` with each sample's own $dl$ through `mc.tearing_parity`; the earlier $dl=1$ value is kept as `T_dl1` and the number of capture labels that change is printed).

In [ ]:
def tearing(ds):
    """(sample, mode) |int A dtheta| / int |A| dtheta, as pyro's compute_linear_tearing_parameter with dl = 1."""
    a = np.asarray(ds.eigenfunctions.sel(field="apar").transpose("sample", "mode", "theta").values)
    th = np.asarray(ds.theta.values, float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.abs(simpson(a, x=th, axis=-1)) / simpson(np.abs(a), x=th, axis=-1)

def dom(ds, v):
    """Value of (sample, mode) array v at each sample's dominant (argmax gamma) mode."""
    g = np.asarray(ds.growth_rate.transpose("sample", "mode").values, float)
    k = np.where(np.isfinite(g), g, -np.inf).argmax(axis=1)
    return v[np.arange(len(k)), k]

D = {}
for dname, (draw, arms) in draws.items():
    gscan, gs2 = mc.load_gs2_cube(gs2_base / draw, "pyro_cube_avg")   # no run directory is read (Fusion_PhD-use8)
    names = [str(n) for n in gs2.sample_name.values]
    gamma_gs2 = np.asarray(gs2.growth_rate.values, float).reshape(len(names), -1)[:, 0]
    unstable = gamma_gs2 > gs2_unstable
    label = np.where(unstable, mc.classify(mc.indicators(gscan, gs2)), "").astype(object)
    rec = dict(names=names, gamma_gs2=gamma_gs2, unstable=unstable, label=label, arm={})
    for arm in arms:
        leaf, sub, kind = arms_all[arm]
        ascan, ds = mc.load_gftm_cube(base / draw / leaf)
        row = {str(n): i for i, n in enumerate(ds.sample_name.values)}
        take = np.array([row.get(n, -1) for n in names])
        have, t = take >= 0, np.maximum(take, 0)
        g = np.asarray(ds.growth_rate.transpose("sample", "mode").values, float)
        om = np.asarray(ds.mode_frequency.transpose("sample", "mode").values, float)
        T1 = tearing(ds)   # dl = 1, the cube-only T of earlier versions
        tp = mc.tearing_parity(ascan, ds, np.flatnonzero(np.isin(np.arange(ds.sizes["sample"]), take[have])))   # pyro's dl, per sample
        Tp = np.asarray(tp["T"].isel(kx=0, drop=True, missing_dims="ignore").transpose("sample", "mode").values, float)
        a = dict(have=have, gamma=np.where(have, dom(ds, g)[t], np.nan), omega=np.where(have, dom(ds, om)[t], np.nan),
                 T=np.where(have, dom(ds, Tp)[t], np.nan), T_dl1=np.where(have, dom(ds, T1)[t], np.nan))
        if "tearing_parameter" in ds:
            Ts = ds.tearing_parameter.isel(kx=0, drop=True, missing_dims="ignore").transpose("sample", "mode").values
            a["T_stored"] = np.where(have, dom(ds, np.asarray(Ts, float))[t], np.nan)
        rec["arm"][arm] = a
        print(dname, arm, "decks", int(have.sum()))
    D[dname] = rec
    lab = label[unstable]
    print(dname, f"{len(names)} cases, GS2-unstable {unstable.sum()}:", {c: int((lab == c).sum()) for c in ("MTM", "KBM", "mixed")})

## Population and the removed count

In [ ]:
for dname, r in D.items():
    n, nu = len(r["names"]), int(r["unstable"].sum())
    lab = r["label"][r["unstable"]]
    nm = int((lab == "MTM").sum())
    r["mtm"] = r["unstable"] & (r["label"] == "MTM")
    print(f"{dname}: {n} cases | GS2-stable {n - nu} | KBM {int((lab == 'KBM').sum())} | mixed {int((lab == 'mixed').sum())} "
          f"| MTM kept {nm} | removed {n - nm} of {n} ({nu - nm} of {nu} GS2-unstable)")

## GFTM $T$ check, and capture

`T` from the stored theta-resolved `apar` against the `tearing_parameter` the GyroRun save job stored (arms A, B). Capture = $\gamma>0$ and $\omega<0$ and $T>0.15$ on the dominant mode.

In [ ]:
for dname, r in D.items():
    for arm, a in r["arm"].items():
        a["mtm_mode"] = a["have"] & (a["gamma"] > 0) & (a["omega"] < 0) & (a["T"] > T_TEAR)
        if "T_stored" in a:
            ok = a["have"] & np.isfinite(a["T_stored"])
            dT = np.abs(a["T"] - a["T_stored"])[ok]
            flip = ((a["T"] > T_TEAR) != (a["T_stored"] > T_TEAR))[ok]
            print(f"{dname} {arm}: |T - T_stored| median {np.median(dT):.2e}, max {dT.max():.2e}; side of 0.15 differs on {int(flip.sum())} of {int(ok.sum())}")
        a["mtm_mode_dl1"] = a["have"] & (a["gamma"] > 0) & (a["omega"] < 0) & (a["T_dl1"] > T_TEAR)
    pop = r["label"] == "MTM"
    for arm, a in r["arm"].items():
        print(f"{dname} {arm}: capture labels that change from dl = 1 to pyro's dl: {int((a['mtm_mode'] != a['mtm_mode_dl1']).sum())} of {len(pop)} cases, "
              f"{int(((a['mtm_mode'] != a['mtm_mode_dl1']) & pop).sum())} of {int(pop.sum())} in the MTM-only population")

## Score every arm on the MTM-only population

medlog = median $|\log_{10}(\gamma_{GFTM}/\gamma_{GS2})|$, medAE = median $|\gamma_{GFTM}-\gamma_{GS2}|$, dominant mode, misses $+\infty$, capture failures scored on their own $\gamma$. 95% bootstrap intervals; the same resampled cases for every arm.

In [ ]:
def score(r, arm, mask):
    a = r["arm"][arm]
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = a["gamma"] / r["gamma_gs2"]
    pos = a["have"] & (a["gamma"] > 0)
    e = np.where(pos, np.abs(np.log10(ratio)), np.inf)
    ae = np.where(pos, np.abs(a["gamma"] - r["gamma_gs2"]), np.inf)
    cap = a["mtm_mode"]
    return dict(e=e[mask], ae=ae[mask], cap=cap[mask], miss=~pos[mask])

rng = np.random.default_rng(seed)
def table(r, arms, mask):
    n = int(mask.sum())
    b = rng.integers(0, n, (n_boot, n))
    out = {}
    for arm in arms:
        s = score(r, arm, mask)
        bm = np.median(s["e"][b], axis=1)
        out[arm] = dict(n=n, medlog=np.median(s["e"]), ci=np.percentile(bm, [2.5, 97.5]), medae=np.median(s["ae"]),
                        capture=s["cap"].mean(), cap_ci=np.percentile(s["cap"][b].mean(axis=1), [2.5, 97.5]),
                        capfail=int((~s["cap"] & ~s["miss"]).sum()), misses=int(s["miss"].sum()))
    return out

T1 = {}
for dname, r in D.items():
    arms = draws[dname][1]
    T1[dname] = table(r, arms, r["mtm"])
    allcase = table(r, arms, r["unstable"])
    print(f"\n{dname}: MTM-only n={int(r['mtm'].sum())} (removed {len(r['names']) - int(r['mtm'].sum())} of {len(r['names'])}); "
          f"all GS2-unstable n={int(r['unstable'].sum())} shown for reference only (superseded headline)")
    print(f"{'arm':18s} medlog [95% CI] | medAE | capture [95% CI] | capture failures | misses || all-case medlog")
    for arm in arms:
        t = T1[dname][arm]
        print(f"{arm:18s} {t['medlog']:.3f} [{t['ci'][0]:.3f},{t['ci'][1]:.3f}] | {t['medae']:.4f} | {t['capture']:.3f} "
              f"[{t['cap_ci'][0]:.3f},{t['cap_ci'][1]:.3f}] | {t['capfail']:3d} | {t['misses']:3d} || {allcase[arm]['medlog']:.3f}")

## Plot

LogMed of the absolute growth-rate error and capture per arm, MTM-only n300. Counts and definitions are in the README.

In [ ]:
out = ROOT / "Plots" / analysis_name
out.mkdir(parents=True, exist_ok=True)
arms = draws["n300"][1]
t = T1["n300"]
x = np.arange(len(arms))
fig1, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(x, [np.log10(t[a]["medae"]) for a in arms], "o")
ax[0].set_ylabel(r"LogMed $|\gamma_{\rm GFTM}-\gamma_{\rm GS2}|$", fontsize=16)
cap = np.array([t[a]["capture"] for a in arms])
ax[1].errorbar(x, cap, yerr=np.array([np.abs(t[a]["cap_ci"] - t[a]["capture"]) for a in arms]).T, ls="", marker="o")
ax[1].set_ylabel("Capture", fontsize=16)
ax[1].set_ylim(0, 1)
for a_ in ax:
    a_.set_xticks(x, arms, rotation=30, ha="right", fontsize=14)
    a_.set_title("NSTX_MTM n300", fontsize=16)
fig1.tight_layout()
plt.show()


## Save

In [ ]:
fig1.savefig(out / "fig1_arms_mtm_only_n300.png")


## Interpretation

_To be written after the executed run._